## Lecture 9: Testing & Documentation

### ****Exercise 1:** Extract and Test**

****Goal:** find a hard-to-test function in your own code; extract the computation core as a pure function; write one test.**

#### ****Step 1:** — identify the problem:**

**Does a function mix computation with timing, printing, or plotting? That is the one to fix. The “Untestable Function” slide is the reference.**

#### ****Step 2:** — extract:**

**Pull the inner pixel computation into a standalone function that takes `c` and `max_iter` and returns an `int`**

#### ****Step 3:** — test:**

- **If your code is already well-structured, extract mandelbrot pixel from your naive or Numba implementation and test it**

- **Choose a test value you can justify mathematically — not one you read off a plot**

****Done?** Run `pytest -v` and confirm the test passes → discuss what “hard to test” looked like in your code.**

### ****Exercise 2:** Add a Docstring**

****Goal:** add a NumPy-style docstring to one Mandelbrot function.**

**Choose a function that:**

- **Does computation (not just timing or plotting)**

- **Has at least two parameters with non-obvious types or semantics**

- **Returns a meaningful value**

**Sections to include (NumPy style):**

1. **One-line summary sentence (ends with a period)**

2. **Parameters — name, type, and description for each parameter**

3. **Returns — type and description of the return value**

4. **Examples (optional but useful)**

****Test your docstring:** read it aloud. Does it tell a new reader everything they need to call the function correctly? Does it state what the return value means, not just its type?**

****Done?** Discuss with a neighbour — compare style choices and what you chose to document.**

### ****Exercise 3 (Optional):** Branch Coverage Report**

****Goal:** generate a branch coverage report and identify untested branches.**

```python
# Install if needed:
mamba install pytest pytest-cov
# Run with branch coverage:
pytest --cov=. --cov-branch --cov-report=term-missing -v
```

```bash
pytest --cov=. --cov-branch --cov-report=term-missing   #"." is current directory
```

**What to look for:**

- **Lines listed under “missing” — never executed by any test**

- **Branch arrows such as `3->5` — the `if` whose `else` path was never taken**

- **In `mandelbrot_pixel`: both sides of the escape condition must be covered — a pixel that escapes and a pixel that does not**

****Done?** Identify one untested branch; write a test that covers it → discuss with a neighbour.**

> NOTE: **ADVANCED EXERCISES** 

```sh
mamba install hypothesis sphinx mutmut
```

### ****Exercise 4 (Optional):** Hypothesis**

****Goal:** Write property tests for two structural invariants of `mandelbrot_pixel`**

#### **Software Quality — Advanced (Optional): Hypothesis**

```python
from hypothesis import given, settings
from hypothesis.strategies import integers

@given(integers())          # generates random integers
def test_abs_non_negative(x):
    assert abs(x) >= 0      # must hold for ANY integer
```

#### **Software Quality — Advanced (Optional): Hypothesis — Mandelbrot**

```python
# Draw random points with |c| <= 3 (covers both inside and outside the set)
@given(complex_numbers(max_magnitude=3.0, allow_nan=False, allow_infinity=False))
@settings(max_examples=200)
def test_result_in_range(c):
    assert 0 <= mandelbrot_pixel(c, 100) <= 100
```

```python
# Draw random points far outside the set (|c| between 3 and 10)
@given(complex_numbers(min_magnitude=3.0, max_magnitude=10.0,
                       allow_nan=False, allow_infinity=False))
def test_outside_set_escapes(c):
    assert mandelbrot_pixel(c, 100) < 100
```

#### **Software Quality — Advanced (Optional): Mutation Testing**

```bash
mamba install mutmut
mutmut run        # inject mutations and run tests
mutmut results    # list surviving mutants
```

```python
assert mandelbrot_pixel(0+2j, 100) == 2   # catches it: mutant returns 1, assertion fails
assert mandelbrot_pixel(0+2j, 100) < 100  # misses it:  mutant returns 1, assertion passes
```

### ****Exercise 5 (Optional):** Performance regression**

****Goal:** Add a performance regression test asserting NumPy is significantly faster than naive**

```python
import time, numpy as np

def time_once(fn, *args):
    t0 = time.perf_counter()
    fn(*args)
    return time.perf_counter() - t0

def test_numpy_faster_than_naive():
    N, MAX_ITER = 64, 100   # small grid: low variance, fast to run
    t_naive = time_once(mandelbrot_naive,  N, MAX_ITER)
    t_numpy = time_once(mandelbrot_numpy,  N, MAX_ITER)
    assert t_numpy < t_naive / 5, (
        f"NumPy ({t_numpy:.4f}s) not 5x faster than naive ({t_naive:.4f}s)"
    )
```

### ****Exercise 6 (Optional):** GitHub Actions CI**

****Goal:** Set up a workflow that runs your test suite automatically on every push**

```yaml
name: Tests
on: [push, pull_request]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.11" }
      - run: pip install numpy pytest pytest-cov
      - run: pytest --cov=. -v
```

### ****Exercise 7 (Optional):** Sphinx**

****Goal:** Generate HTML documentation from your docstrings**

### ****Exercise 8 (Optional):** mutmut mutation testing**

****Goal:** Run mutation testing; identify surviving mutants; add tests to kill them**